# 9장. 회귀 분석으로 숫자 예측하기

이 Notebook의 핵심은 높은 점수를 만드는 것이 아니라 다음 순서를 지키는 것입니다.

`예측 시점 → 누수 방지 → 시간 분할 → Train TimeSeriesSplit → 모델 선택 고정 → Final Test → Baseline 비교`

**Final Test 결과를 보고 모델을 고르지 않습니다.**

## 0. 실행 전 확인

Chapter09는 Chapter05의 오류 탐지용 전용 Raw가 아니라 **공통 프로젝트 `data/raw`**를 사용합니다. 먼저 프로젝트 루트에서 다음 명령을 실행해 관계 검증을 통과한 모델링 입력을 준비합니다.

```powershell
python scripts/prepare_ch09_data.py
```

이 명령은 공통 `data/raw`를 전처리하고 FK 관계를 검증한 뒤 `data/processed/*_clean.csv`를 생성합니다.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'

from src.regression import (
    FEATURE_COLUMNS,
    FORBIDDEN_FEATURES,
    TARGET_COLUMN,
    build_feature_audit,
    build_leakage_checklist,
    build_regression_dataset,
    build_regression_validation,
    build_split_summary,
    create_prediction_result,
    cross_validate_regression_models,
    load_regression_source_data,
    make_regression_models,
    run_regression_analysis,
    save_regression_outputs,
    select_diagnostic_model,
    split_model_data_by_time,
    train_and_evaluate_models,
    validate_feature_columns,
)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('PROCESSED_DIR:', PROCESSED_DIR)

## 1. 예측 문제와 예측 시점

- Target: 주문 1건의 주문 상세 금액 합계 `order_total`
- 예측 시점: 주문 메타데이터와 고객의 비식별 특성은 알 수 있지만 주문 상세 수량·단가·금액은 모델에 제공하지 않는 시점
- 목표 계산 재료와 사후 정보, 식별자는 feature에서 제외합니다.

In [ ]:
print('Target:', TARGET_COLUMN)
print('Allowed features:', FEATURE_COLUMNS)
print('Forbidden features:', sorted(FORBIDDEN_FEATURES))
validate_feature_columns()
print('Feature leakage gate: PASS')

## 2. 검증된 전처리 데이터를 불러옵니다

In [ ]:
data = load_regression_source_data(PROCESSED_DIR)
for name, frame in data.items():
    print(name, frame.shape)

## 3. 주문 단위 Target을 만들고 관계를 검증합니다

`line_total = quantity × unit_price`를 확인하고, 주문과 Target은 `one_to_one`, 주문과 고객은 `many_to_one` 관계가 깨지지 않는지 확인합니다.

In [ ]:
model_data = build_regression_dataset(
    customers=data['customers'],
    orders=data['orders'],
    order_items=data['order_items'],
)
display(model_data.head())
print('Model rows:', len(model_data))

## 4. Feature Audit을 확인합니다

In [ ]:
feature_audit = build_feature_audit()
display(feature_audit)

## 5. 시간 순서로 Train / Final Test를 분리합니다

같은 달력 날짜를 양쪽에 나누지 않습니다. Final Test는 모델 선택이 끝날 때까지 사용하지 않습니다.

In [ ]:
train_data, test_data = split_model_data_by_time(model_data, test_size=0.2)
split_summary = build_split_summary(train_data, test_data)
display(split_summary)
assert train_data['order_date'].max().normalize() < test_data['order_date'].min().normalize()
print('Chronological split gate: PASS')

In [ ]:
X_train = train_data[FEATURE_COLUMNS].copy()
X_test = test_data[FEATURE_COLUMNS].copy()
y_train = train_data[TARGET_COLUMN].copy()
y_test = test_data[TARGET_COLUMN].copy()
print('Train:', X_train.shape, 'Final Test:', X_test.shape)

## 6. Pipeline 후보 모델을 준비합니다

숫자형 결측 대체·표준화와 범주형 결측 대체·One-Hot Encoding은 모델 Pipeline 안에서 각 Train Fold로만 학습됩니다.

In [ ]:
models = make_regression_models(random_state=42)
print(list(models))

## 7. Train 기간의 TimeSeriesSplit으로 후보를 비교합니다

여기까지는 Final Test의 `y_test`를 모델 선택에 사용하지 않습니다.

In [ ]:
cv_summary = cross_validate_regression_models(
    models=models,
    X_train=X_train,
    y_train=y_train,
)
display(cv_summary)

## 8. 비베이스라인 모델 선택을 고정합니다

Linear Regression과 Random Forest 중 Train CV의 평균 MAE가 더 낮은 모델을 선택합니다. **이 시점까지 Final Test 성능은 보지 않습니다.**

In [ ]:
selected_model_name = select_diagnostic_model(cv_summary)
print('Frozen selected model:', selected_model_name)

## 9. 이제 Final Test에서 Baseline과 고정 모델만 평가합니다

Final Test 결과가 기대와 다르더라도 같은 Test를 보고 다른 후보로 교체하지 않습니다.

In [ ]:
model_comparison, predictions = train_and_evaluate_models(
    models=models,
    selected_model_name=selected_model_name,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
)
display(model_comparison)

### 지표 읽기

- MAE: 평균적으로 얼마만큼 틀리는가
- RMSE: 큰 오차에 더 민감
- R²: 평균 예측 대비 설명력, 음수가 될 수 있음
- MAE 개선율: Dummy baseline 대비 고정 모델의 Final Test MAE 개선 정도

## 10. 실제값·예측값·잔차를 진단합니다

In [ ]:
prediction_result = create_prediction_result(
    test_data=test_data,
    y_test=y_test,
    y_pred=predictions[selected_model_name],
    model_name=selected_model_name,
)
display(prediction_result.head(10))
print('주의: order_id가 포함된 이 표는 내부 진단용입니다.')

## 11. 자동 Validation Evidence를 만듭니다

In [ ]:
validation = build_regression_validation(
    train_data=train_data,
    test_data=test_data,
    cv_summary=cv_summary,
    selected_model_name=selected_model_name,
    model_comparison=model_comparison,
)
display(validation)
assert validation['status'].eq('PASS').all()

## 12. Evidence·내부 진단·공개 보고서를 구분해 저장합니다

In [ ]:
checklist = build_leakage_checklist()
output_paths = save_regression_outputs(
    model_data=model_data,
    split_summary=split_summary,
    feature_audit=feature_audit,
    cv_summary=cv_summary,
    selected_model_name=selected_model_name,
    model_comparison=model_comparison,
    prediction_result=prediction_result,
    validation=validation,
    checklist=checklist,
    report_dir=REPORT_DIR,
)
for name, path in output_paths.items():
    print(name, '->', path)

## 13. 전체 공통 함수를 새 실행으로 다시 확인합니다

셀의 메모리 상태가 아니라 공통 함수가 같은 순서를 재현하는지 확인합니다.

In [ ]:
result = run_regression_analysis(
    processed_dir=PROCESSED_DIR,
    report_dir=REPORT_DIR,
    test_size=0.2,
    random_state=42,
)
print('Selected by Train CV:', result['selected_model_name'])
display(result['validation'])

## 14. 해석

낮은 성능이나 음수 R²도 숨기지 않습니다. 현재 예측 시점에 허용한 정보가 약하다면 `현재 데이터로는 모델 사용 보류`도 올바른 결론입니다.

LLM이 만든 코드도 같은 예측 시점·누수·Pipeline·Train CV·Final Test 보호 기준으로 검토합니다.

## 15. 완료 체크

- [ ] `python scripts/prepare_ch09_data.py`로 공통 모델링 입력을 준비했다.
- [ ] Target과 예측 시점을 설명할 수 있다.
- [ ] 금지 feature가 입력에 없음을 확인했다.
- [ ] 같은 날짜가 Train과 Final Test에 동시에 들어가지 않는다.
- [ ] 전처리는 Pipeline 내부에서 학습된다.
- [ ] Train TimeSeriesSplit으로 후보를 비교했다.
- [ ] Final Test 전에 Selected Model을 고정했다.
- [ ] Final Test에서는 Baseline과 Frozen Model만 비교했다.
- [ ] MAE, RMSE, R²를 함께 읽었다.
- [ ] 내부 식별자 결과와 공개 보고서를 구분했다.
- [ ] Validation Evidence가 모두 PASS다.

## 16. 대표 오차 사례 진단 (Internal)

아래 표는 `order_id`가 포함된 내부 진단용입니다. 큰 오차 사례의 주문 상태와 주문 상세 구성을 확인하기 위한 것이며, 이 컬럼들은 **모델 입력에 사용하지 않습니다.**

In [ ]:
item_summary = (
    data['order_items']
    .groupby('order_id')
    .agg(n_lines=('order_item_id', 'count'), total_qty=('quantity', 'sum'))
    .reset_index()
)
case_view = (
    prediction_result
    .merge(test_data[['order_id', 'order_status', 'payment_method', 'gender', 'age', 'city']], on='order_id', how='left', validate='one_to_one')
    .merge(item_summary, on='order_id', how='left', validate='one_to_one')
)
case_view['predicted_order_total'] = case_view['predicted_order_total'].round(0)
case_view['residual'] = case_view['residual'].round(0)
case_view['abs_error'] = case_view['abs_error'].round(0)
print('Train target mean:', round(y_train.mean(), 0))
print('Final Test prediction range:', round(prediction_result['predicted_order_total'].min(), 0), '~', round(prediction_result['predicted_order_total'].max(), 0))
print('Final Test actual range:', y_test.min(), '~', y_test.max())
display(case_view.sort_values('abs_error', ascending=False).head(10))

# Chapter 09 답안. 회귀 분석으로 숫자 예측하기

## 제출 정보
- 이름: 김혜빈
- GitHub ID: Chloe-Hyebin-Kim
- 작성일: 2026-11-11
- 최종 Notebook URL: https://github.com/Chloe-Hyebin-Kim/llm-data-analysis-course/blob/main/practice/chapter09/chapter09.ipynb

## 1. 예측 문제 정의
- Target: `order_total` (같은 `order_id`의 `line_total` 합계)
- 예측 단위: 주문 1건
- 예측 시점: 주문이 생성되어 주문일·결제 수단과 고객 정보는 확인되었지만, 주문 상세(상품·수량·단가)와 주문 상태는 아직 쓰지 않는 시점 (교육용 가정)
- 예측 시점에 알 수 있는 정보: `order_month`, `order_dayofweek`(주문일에서 계산), `payment_method`, 고객의 비식별 특성 `age`, `gender`, `city`
- 예측 시점에 알 수 없는 정보: 주문 상세의 `quantity`, `unit_price`, `line_total`과 여기서 만든 집계(`item_count`, `total_quantity`, `avg_unit_price`), 이후 확정되는 `order_status`

### 나의 해석과 판단
주문 상세가 확정되면 `order_total`은 계산으로 바로 알 수 있으므로 예측할 필요가 없습니다. 예측이 의미가 있으려면 주문 상세를 보기 전 시점이어야 하고, 그 시점에 실제로 알 수 있는 정보만 feature로 써야 합니다. 그래서 주문 메타데이터와 고객 비식별 특성만 아는 시점을 예측 시점으로 정했습니다. 단, `payment_method`가 상세보다 먼저 확정된다는 것은 교육용 가정이며, 실제 업무에서는 결제 흐름을 확인해야 합니다.

## 2. Feature Leakage Audit

| feature | 예측 시점 사용 가능 | Target 계산 재료 | 사후 정보/ID | 최종 사용 | 판단 이유 |
| --- | --- | --- | --- | --- | --- |
| order_month | 예 | 아니오 | 아니오 | 사용 | 주문일에서 계산, 주문 시점에 알 수 있음 |
| order_dayofweek | 예 | 아니오 | 아니오 | 사용 | 주문일에서 계산, 주문 시점에 알 수 있음 |
| age | 예 | 아니오 | 아니오 | 사용 | 고객 비식별 특성 |
| payment_method | 예(교육용 가정) | 아니오 | 아니오 | 사용 | 주문 메타데이터 |
| gender | 예 | 아니오 | 아니오 | 사용 | 고객 비식별 특성 |
| city | 예 | 아니오 | 아니오 | 사용 | 고객 비식별 특성 |
| order_total | - | 예측 대상 자체 | - | 금지 | Target을 입력에 넣으면 정답을 그대로 주는 것 |
| line_total | 아니오 | 예 | - | 금지 | Target을 구성하는 주문 상세 금액 |
| quantity | 아니오 | 예 | - | 금지 | Target 계산 재료 |
| unit_price | 아니오 | 예 | - | 금지 | Target 계산 재료 |
| item_count | 아니오 | 아니오 | 사후 집계 | 금지 | 주문 상세 확인 뒤에 계산되는 집계 |
| total_quantity | 아니오 | 간접(대리 변수) | 사후 집계 | 금지 | 주문 상세에서 만든 Target 대리 변수 |
| avg_unit_price | 아니오 | 간접(대리 변수) | 사후 집계 | 금지 | 주문 상세에서 만든 Target 대리 변수 |
| order_status | 아니오 | 아니오 | 사후 정보 | 금지 | 예측 시점 이후 확정될 수 있는 상태 |
| order_id | - | 아니오 | ID | 금지 | 주문 식별자, 의미 없는 패턴 학습 위험 |
| customer_id | - | 아니오 | ID | 금지 | 고객 식별자 |
| product_id | 아니오 | 아니오 | ID | 금지 | 주문 상세가 확인되어야 알 수 있는 식별자 |

- 금지 feature overlap 개수: 0 (`Feature leakage gate: PASS`, Validation `forbidden_feature_overlap = 0`)
- 가장 위험하다고 판단한 feature: `line_total` (그다음 `quantity`, `unit_price`)
- 이유: `order_total`이 `line_total`의 합이라, 넣으면 모델이 예측이 아니라 덧셈을 학습해 점수가 비정상적으로 높아집니다. 예측 시점에는 존재하지 않는 값이라 실제 사용 환경에서 재현되지 않습니다.

![Leakage 점검](images/step02_leakage.png)

## 3. Target 생성과 관계 검증

### 계산 기준

```text
line_total = quantity × unit_price
order_total = order_id별 line_total 합계
```

- line_total 검증 결과: 불일치 0건 (`build_order_totals`는 불일치가 1건이라도 있으면 `ValueError`로 멈추는데, 정상 완료됨)
- orders ↔ target 관계 검증 결과: `one_to_one` 통과, 연결 안 된 주문 0건, 모델링 행 300 = orders 300행
- orders → customers 관계 검증 결과: `many_to_one` 통과, customers에 연결되지 않는 주문 0건

입력 준비(`prepare_ch09_data.py`)의 FK 검증도 3개 항목 모두 `invalid_count = 0`이었습니다.

### 나의 해석과 판단
관계를 확인하지 않고 병합만 하면 키가 맞지 않는 행이 inner join에서 조용히 빠지거나, 키 중복 때문에 행이 늘어나 같은 주문의 금액이 두 번 들어갈 수 있습니다. 그러면 모델은 실제와 다른 데이터로 학습하고, 결과 표에는 아무 경고도 나타나지 않습니다. 그래서 `validate`와 연결 건수 확인으로 행 손실이나 중복이 있으면 멈추게 하는 것이 맞다고 판단합니다.

## 4. Train / Final Test 분할

- Train 시작일: 2025-09-15
- Train 종료일: 2026-06-20
- Train 행 수: 244 (81.33%)
- Final Test 시작일: 2026-06-21
- Final Test 종료일: 2026-09-14
- Final Test 행 수: 56 (18.67%)
- 같은 달력 날짜 중복 여부: 없음 (Train 마지막 날 2026-06-20 < Final Test 첫날 2026-06-21, `Chronological split gate: PASS`)

비율이 정확히 80:20이 아닌 이유는 날짜 단위로 나누어 같은 날의 주문을 한쪽에만 넣기 때문입니다.

### 분할 판단
이 모델은 과거 주문으로 앞으로 들어올 주문 금액을 예측하는 용도입니다. 랜덤 분할을 하면 미래 주문이 Train에 섞여 실제 사용 상황보다 성능이 좋게 나올 수 있습니다. 시간 순서로 나누면 "과거만 알고 미래를 예측한다"는 실제 조건을 그대로 평가할 수 있습니다.

![시간 분할](images/step04_split.png)

## 5. Train TimeSeriesSplit 후보 비교

| 모델 | CV MAE 평균 | CV MAE 표준편차 | CV R² 평균 | 비고 |
| --- | ---: | ---: | ---: | --- |
| Baseline Mean | 423,005 | 44,153 | -0.023 | 비교 기준 |
| Linear Regression | 476,605 | 74,179 | -0.393 | MAE 가장 높음, 변동 가장 큼 |
| Random Forest | 434,989 | 23,876 | -0.111 | 비베이스라인 중 MAE 낮음, 변동 가장 작음 |

(5-fold, 원 단위 반올림)

### Train CV로 선택한 비베이스라인 모델
- Selected Model: Random Forest
- 선택 기준: 비베이스라인 후보(Linear Regression, Random Forest) 중 `cv_MAE_mean`이 더 낮은 모델 (434,989 < 476,605)
- Final Test 결과를 보기 전에 선택했는가: 예 (`select_diagnostic_model(cv_summary)`로 고정한 뒤 Final Test 평가)

### 나의 해석과 판단
Train CV에서도 Random Forest(434,989)가 Baseline Mean(423,005)보다 MAE가 11,984원 높고, 세 모델 모두 CV R²가 음수입니다. Final Test 전부터 현재 feature로는 평균 예측을 넘기 어렵다는 신호가 있었습니다.

표준편차를 함께 보는 이유는 평균이 비슷해도 기간에 따라 성능이 크게 흔들리는 모델은 새 기간에서 결과를 예상하기 어렵기 때문입니다. Linear Regression은 평균도 가장 나쁘고 표준편차(74,179)도 가장 커서 불안정합니다. Random Forest는 표준편차가 23,876으로 가장 작아 기간별로는 비교적 일관되지만, 일관되게 baseline보다 조금 나쁜 수준입니다.

## 6. Final Test: Baseline vs Frozen Model

| 모델 | selection role | Train MAE | Test MAE | Test RMSE | Test R² | Baseline 대비 MAE 개선율 |
| --- | --- | ---: | ---: | ---: | ---: | ---: |
| Baseline Mean | baseline | 416,266 | 443,450 | 522,190 | -0.001 | 0.00% |
| Frozen Selected Model (Random Forest) | selected_by_train_cv | 327,646 | 464,694 | 550,432 | -0.112 | -4.79% |

![최종 회귀 평가](images/step06_final_metrics.png)

### 결과 관찰
- 개선율 계산: (443,450 − 464,694) ÷ 443,450 × 100 = **-4.79%**. Frozen Model이 baseline보다 MAE가 21,244원 더 큽니다.
- Random Forest의 Train MAE 327,646 → Test MAE 464,694로 137,048원 차이가 납니다. Train에는 맞췄지만 새 기간에는 일반화되지 않았습니다.
- Final Test 예측값 범위는 약 541,590~1,082,731원인데, 실제값 범위는 10,000~1,877,000원입니다. 예측이 평균 근처에 몰려 있습니다.

### 나의 해석과 판단
- **Frozen Model이 baseline보다 실제로 개선되었는가?** 아니요. Test MAE, RMSE, R² 모두 baseline보다 나쁩니다. Train CV 결과(baseline보다 MAE 높음)와 같은 방향이라 Final Test만의 우연한 결과로 보기 어렵습니다. 결과가 나쁘다고 Final Test를 보고 다른 후보로 바꾸지 않았습니다.
- **MAE와 RMSE 차이:** RMSE/MAE 비율은 Frozen Model 550,432 ÷ 464,694 ≈ 1.18, baseline 522,190 ÷ 443,450 ≈ 1.18입니다. 한두 건의 극단 오차가 전체를 끌어올렸다기보다 오차가 전반적으로 크다고 봅니다. 다만 7번처럼 100만 원이 넘는 오차도 있습니다.
- **R² 해석:** baseline R²(-0.001)는 Train 평균으로 Test 평균을 거의 맞혔다는 뜻이고, Frozen Model R²(-0.112)는 Test 기간의 평균값 하나로 예측하는 것보다도 제곱 오차가 크다는 뜻입니다. 코드 오류가 아니라 허용된 feature에 주문 금액을 설명하는 정보가 거의 없다는 결과로 해석합니다.

### 업무·분석적 의미
Final Test 실제 주문 금액 평균이 약 837,821원인데 평균 오차가 약 46만 원이면, 주문 1건 금액을 예측하는 용도로는 쓸 수 없습니다. 지금은 "주문 금액은 평균 약 85만 원 수준"이라는 baseline 이상의 정보를 주지 못합니다.

## 7. 대표 오차 사례

16번 셀(Internal 진단, 절대 오차 상위 10건) 기준입니다. 사례 1은 1위, 사례 2는 7위, 사례 3은 9위입니다. 금액 단위는 원이며 예측값은 반올림했습니다.

### 사례 1
- 실제값: 1,877,000
- 예측값: 541,590
- residual: +1,335,410 (낮게 예측)
- absolute error: 1,335,410 (Final Test 최대)
- 관찰: 주문 상세 3행, 총 수량 13개로 금액이 큰 주문입니다. 예측값은 Final Test 예측 범위의 하단에 가깝습니다. 주문 상태는 cancelled입니다.
- 가능한 이유 후보: 금액을 크게 만든 수량·단가는 예측 시점에 사용할 수 없는 정보라 모델이 알 수 없습니다. 허용 feature(월, 요일, 결제 수단, 성별, 나이, 도시)만으로는 고액 주문을 구분할 신호가 부족했을 수 있습니다.
- 추가 확인: 같은 고객의 과거 주문 금액처럼 예측 시점 이전에 알 수 있는 이력 feature가 고액 주문을 구분하는 데 도움이 되는지 확인합니다.

### 사례 2
- 실제값: 10,000
- 예측값: 852,718
- residual: -842,718 (높게 예측)
- absolute error: 842,718
- 관찰: 주문 상세 1행(단가 5,000원 × 2개)인 소액 주문인데, 예측은 Train 평균(약 855,295원)과 비슷합니다. 주문 상태는 cancelled입니다.
- 가능한 이유 후보: 모델 예측이 평균 근처에 몰려 있어 소액 주문을 구분하지 못한 것으로 보입니다.
- 추가 확인: 취소 주문을 모델링 대상에 포함하는 것이 업무 목적에 맞는지 확인합니다. 단, `order_status`는 사후 정보라 feature로는 쓰지 않습니다.

### 사례 3
- 실제값: 257,000
- 예측값: 1,082,731
- residual: -825,731 (높게 예측)
- absolute error: 825,731
- 관찰: Final Test 예측값 중 가장 높은 값이 실제로는 소액 주문이었습니다. 주문 상태는 completed입니다.
- 가능한 이유 후보: 이 주문의 feature 조합이 Train의 고액 주문과 비슷했을 수 있으나, 데이터가 직접 보여 주지 않아 단정할 수 없습니다.
- 추가 확인: Random Forest의 feature 중요도와, 같은 feature 조합을 가진 Train 주문의 금액 분포를 확인합니다.

> 위 이유는 관찰을 바탕으로 한 후보이며, 데이터가 원인을 직접 보여 주지는 않습니다.

## 8. Validation Evidence

`reports/ch09_regression_validation.csv` 기준입니다.

| check | value | status |
| --- | --- | --- |
| forbidden_feature_overlap | 0 | PASS |
| strict_train_before_test | True | PASS |
| selected_model_exists_in_train_cv | True | PASS |
| final_test_contains_baseline | True | PASS |
| final_test_contains_frozen_selected_model | True | PASS |
| test_rows_for_r2 | 56 | PASS |

### FAIL 항목
- 없음
- Validation이 모두 PASS인 것은 **절차가 올바르다**는 뜻이지 모델 성능이 좋다는 뜻은 아닙니다. 성능 판단은 6번과 10번에 따로 적었습니다.

## 9. Evidence와 공개 범위

### 생성된 Evidence
- [x] `ch09_regression_split_summary.csv`
- [x] `ch09_regression_feature_audit.csv`
- [x] `ch09_regression_cv_summary.csv`
- [x] `ch09_regression_model_comparison.csv`
- [x] `ch09_regression_validation.csv`
- [x] `ch09_regression_checklist.csv`

### Internal 파일
- [x] `ch09_regression_model_data_internal.csv`
- [x] `ch09_regression_predictions_internal.csv`

### 공개 결과
- [x] `ch09_regression_report.md`
- [x] `ch09_actual_vs_predicted.png`
- [x] `ch09_residual_histogram.png`

### 공개 범위 판단
`order_id`가 있는 예측 결과는 다른 데이터와 연결하면 특정 주문과 고객을 다시 찾아낼 수 있습니다. 또 "이 주문은 얼마로 예측했고 얼마나 틀렸다"는 개별 정보가 그대로 드러납니다. 그래서 주문별 결과는 `_internal` 파일로 분리해 내부 진단에만 쓰고, 공개 보고서에는 집계 지표와 그래프만 둡니다. 이 Notebook의 오차 사례 설명에도 `order_id`는 적지 않았습니다.

## 10. 최종 사용 판단

- [ ] 현재 수준에서 참고용 사용 가능
- [ ] 추가 검증 후 사용 가능
- [x] 현재 데이터로는 사용 보류

### 판단 근거
1. **Baseline보다 나쁨:** Final Test MAE가 baseline보다 4.79% 높고(464,694 vs 443,450), RMSE와 R²도 baseline보다 나쁩니다.
2. **Train CV에서도 같은 결과:** 선택된 Random Forest의 CV MAE(434,989)가 baseline(423,005)보다 높고, 세 후보 모두 CV R²가 음수입니다. 시점을 바꿔도 평균 예측을 넘지 못했습니다.
3. **절차는 올바름:** 누수 feature 0개, 날짜 기준 엄격한 시간 분할, Final Test 전 모델 고정, Validation 6개 항목 모두 PASS입니다. 그래서 낮은 성능은 절차 오류가 아니라 **현재 허용된 feature의 정보 부족**으로 판단합니다.

### 업무적 위험
이 모델로 주문 금액을 예측해 재고나 매출 계획에 쓰면, 평균 약 46만 원의 오차가 생기고 고액 주문은 크게 낮게, 소액 주문은 크게 높게 예측합니다. 평균값 하나를 쓰는 것보다 오히려 판단을 더 틀리게 만들 수 있습니다.

### 현재 데이터의 한계
- 전체 300건, Final Test 56건으로 데이터가 적어 지표가 표본에 따라 흔들릴 수 있습니다.
- 허용 feature 6개는 주문 금액과 직접 관련된 정보(무엇을 얼마나 샀는지)를 담고 있지 않습니다.
- completed 외에 cancelled·refunded 주문도 Target에 함께 들어 있습니다.
- 교육용으로 생성한 데이터라 실제 고객 행동 패턴을 반영한다고 볼 수 없습니다.

### 다음 개선 우선순위
1. 예측 시점 이전에 알 수 있는 고객 이력 feature(과거 주문 수·과거 평균 주문 금액 등)를 시점을 지켜 만들고, 누수 점검부터 다시 수행합니다.
2. 데이터 기간·건수를 늘려 Train CV와 Final Test의 안정성을 다시 확인합니다.
3. 업무 목적에 따라 completed 주문만 대상으로 할지, 취소·환불 주문을 포함할지 기준을 정합니다.

## 11. 전체 재실행 확인

실행 명령:

```powershell
python scripts/prepare_ch09_data.py
python scripts/run_regression_analysis.py
```

- 실행 성공 여부: 성공 (입력 준비 FK 검증 3개 모두 0건, 회귀 분석 완료)
- Train CV Selected Model: Random Forest
- Final Validation 전체 PASS 여부: 6개 항목 모두 PASS
- Notebook 결과와 스크립트 결과의 일관성: 분할(244/56), CV 요약, Selected Model, Final Test 지표, Validation이 Notebook 결과와 같습니다. Notebook 13번 셀의 `run_regression_analysis()` 새 실행도 Random Forest와 전체 PASS로 같습니다.

## 최종 체크

- [x] Target과 예측 시점을 정의했습니다.
- [x] Leakage Audit을 수행했습니다.
- [x] Target 계산과 관계 검증을 확인했습니다.
- [x] Train과 Final Test가 시간 순서로 엄격히 분리되었습니다.
- [x] 전처리가 Pipeline 내부에서 학습됩니다.
- [x] Train TimeSeriesSplit으로 후보를 비교했습니다.
- [x] Final Test 전에 Selected Model을 고정했습니다.
- [x] Final Test에서는 Baseline과 Frozen Model만 비교했습니다.
- [x] MAE, RMSE, R²를 함께 해석했습니다.
- [x] 대표 오차 사례를 관찰과 가설로 구분했습니다.
- [x] Validation Evidence가 모두 PASS입니다.
- [x] Internal 결과와 공개 결과를 구분했습니다.
- [x] 낮은 성능도 숨기지 않았습니다.
- [x] 전체 스크립트를 재실행했습니다.
- [x] 최종 Notebook URL을 제출합니다.